In [1]:
import os
import csv
import shutil
import requests
from tqdm import tqdm
from PIL import Image
from io import BytesIO

import aiohttp
import asyncio


from ddgs import DDGS
import time

In [2]:
import sys
import pandas as pd
from pathlib import Path

# Make the backend src discoverable regardless of where the notebook is run from
_backend_src = Path("../backend/app/src").resolve()
if str(_backend_src) not in sys.path:
    sys.path.insert(0, str(_backend_src))

from herbs_detection.model import predict_top3 as pt_top3
from herbs_detection.model_sklearn import predict_top3 as sk_top3

print("Both models loaded OK")

Both models loaded OK


In [ ]:
plants = {
    "basil": "Ocimum basilicum",
    "dill": "Anethum graveolens",
    "sage": "Salvia officinalis"
}   

#plants = {
#    
#    "basil": "Ocimum basilicum",
#}
#    "lemon_verbena": "Aloysia citrodora"
# "coriander": "Coriandrum sativum",
    #"thyme": "Thymus vulgaris",
#    "chives": "Allium schoenoprasum",
#    "savory": "Satureja hortensis",
#    "dill": "Anethum graveolens",
#    "coriander": "Coriandrum sativum",
#    "angelica": "Angelica archangelica",
#    "oregano": "Origanum vulgare",
#    "fennel": "Foeniculum vulgare",  
#    "tarragon": "Artemisia dracunculus",
#    "parsley": "Petroselinum crispum",
#    "mint": "Mentha",
#    "chamomile": "Matricaria chamomilla",
#    "lovage": "Levisticum officinale",
#    "lavender": "Lavandula",
#    "mugwort": "Artemisia vulgaris",
##    "rosemary": "Salvia rosmarinus",
#    "sage": "Salvia officinalis",
#    "lemongrass": "Cymbopogon citratus",
#    "bay_leaf": "Laurus nobilis",
#    "marjoram": "Origanum majorana",
#    "savory": "Satureja hortensis",
#    "anise": "Pimpinella anisum",
#    "stevia": "Stevia rebaudiana",
#    "lemon_balm": "Melissa officinalis",
#    "pandan": "Pandanus amaryllifolius",
#    "holy_basil": "Ocimum tenuiflorum",
#    "lemon_verbena": "Aloysia citrodora",
#    "wintergreen": "Gaultheria procumbens",
#    "hyssop": "Hyssopus officinalis",   
#    "borage": "Borago officinalis"
#}



In [ ]:
# ---------------------------
# TELECHARGEMENT + RESIZE
# ---------------------------

def download_and_resize(url, filepath):

    try:
        r = requests.get(url, timeout=15)

        img = Image.open(BytesIO(r.content)).convert("RGB")

        img = img.resize(IMAGE_SIZE)

        img.save(filepath, "JPEG", quality=90)

        return True

    except:
        return False

async def download_and_resize_async(session, url, filepath, timeout=10):
    """Download and resize a single image asynchronously."""
    try:
        async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout)) as resp:
            if resp.status == 200:
                img = Image.open(BytesIO(await resp.read())).convert("RGB")
                img = img.resize(IMAGE_SIZE)
                img.save(filepath, "JPEG", quality=90)
                return True
    except Exception as e:
        print(f"    [ERROR] {str(e)[:60]}", end="")
    return False

async def download_batch_async(urls_filepaths, max_concurrent=10):
    """
    Download multiple images concurrently.
    
    Args:
        urls_filepaths: list of (url, filepath) tuples
        max_concurrent: max parallel downloads (10–20 is good; don't exceed ~50)
    
    Returns:
        count of successful downloads
    """
    success = 0
    semaphore = asyncio.Semaphore(max_concurrent)
    
    async def bounded_download(session, url, filepath):
        nonlocal success
        async with semaphore:
            if await download_and_resize_async(session, url, filepath):
                success += 1
            return success
    
    connector = aiohttp.TCPConnector(limit_per_host=5)
    async with aiohttp.ClientSession(connector=connector) as session:
        tasks = [bounded_download(session, url, fp) for url, fp in urls_filepaths]
        await asyncio.gather(*tasks)
    
    return success

In [ ]:
# ---------------------------
# PARAMETRES
# ---------------------------

DATASET_DIR      = "../data/dataset_aromatic_plants"


IMAGES_PER_PLANT = 100   # number of correctly-predicted images to SAVE per plant
IMAGE_SIZE       = (400, 400)
BATCH_SIZE       = 20    # images to download per round before predicting

# ---------------------------
# CREATION DOSSIERS
# ---------------------------

os.makedirs(DATASET_DIR, exist_ok=True)
csv_rows = []

for plant_name, taxon in plants.items():
    print(f"\n{'='*60}")
    print(f"Downloading: {plant_name} ({taxon})")

    plant_dir = os.path.join(DATASET_DIR, f"{plant_name}5")
    tmp_dir   = os.path.join(DATASET_DIR, f"_tmp_{plant_name}")
    os.makedirs(plant_dir, exist_ok=True)
    os.makedirs(tmp_dir,   exist_ok=True)

    saved_count   = 0   # correctly-predicted images saved so far
    url_idx       = 0   # pointer into all_urls
    page          = 1
    all_urls      = []
    all_dates     = []
    no_more_pages = False

    pbar = tqdm(total=IMAGES_PER_PLANT, desc=f"  Saving {plant_name}")

    while saved_count < IMAGES_PER_PLANT:

        # ── 1. Fetch more URLs from the API until we have a full batch ──────
        while len(all_urls) - url_idx < BATCH_SIZE and not no_more_pages:
            api_url = "https://api.inaturalist.org/v1/observations"
            params  = {
                "taxon_name":    taxon,
                "quality_grade": "casual,needs_id,research",
                "identified":    "true",
                "photos":        "true",
                "per_page":      200,
                "page":          page,
                "month":         "4,5,6,7,8,9,10",
                "year":          "2020,2021,2022,2023,2024",
                "order":         "asc",
            }
            results = requests.get(api_url, params=params).json().get("results", [])
            if not results:
                no_more_pages = True
                break
            for obs in results:
                for photo in obs["photos"]:
                    all_urls.append(photo["url"].replace("square", "large"))
                    all_dates.append(obs["observed_on_details"]["date"])
            page += 1

        # ── 2. Build the next batch ─────────────────────────────────────────
        batch_end   = min(url_idx + BATCH_SIZE, len(all_urls))
        batch_urls  = all_urls [url_idx:batch_end]
        batch_dates = all_dates[url_idx:batch_end]

        if not batch_urls:
            print(f"  No more URLs available. Saved {saved_count}/{IMAGES_PER_PLANT}")
            break

        tmp_paths      = [
            os.path.join(tmp_dir, f"tmp_{url_idx + i:05d}.jpg")
            for i in range(len(batch_urls))
        ]
        urls_filepaths = list(zip(batch_urls, tmp_paths))

        # ── 3. Async download of the batch ──────────────────────────────────
        await download_batch_async(urls_filepaths, max_concurrent=10)

        # ── 4. Predict (both models) & filter ───────────────────────────────
        for tmp_path, date_val in zip(tmp_paths, batch_dates):
            if not os.path.exists(tmp_path):
                continue

            try:
                pt_pred1 = plant_name #pt_top3(tmp_path)[0][0]   # PyTorch  top-1
                sk_pred1 = plant_name #sk_top3(tmp_path)[0][0]   # Sklearn  top-1
            except Exception as e:
                print(f"\n  [PRED ERROR] {e}")
                os.remove(tmp_path)
                continue
            
            if pt_pred1 == plant_name and sk_pred1 == plant_name:
                final_path = os.path.join(plant_dir, f"{plant_name}_2{saved_count:04d}.jpg")
                os.rename(tmp_path, final_path)
                csv_rows.append([plant_name,
                                  os.path.join(plant_name, os.path.basename(final_path)),
                                  date_val])
                saved_count += 1
                pbar.update(1)
                if saved_count >= IMAGES_PER_PLANT:
                    break
            else:
                os.remove(tmp_path)

        url_idx = batch_end

        if no_more_pages and url_idx >= len(all_urls):
            print(f"  Exhausted all available URLs. Saved {saved_count}/{IMAGES_PER_PLANT}")
            break

    pbar.close()
    shutil.rmtree(tmp_dir, ignore_errors=True)
    print(f"  ✓ {saved_count}/{IMAGES_PER_PLANT} verified images saved for {plant_name}")

    CSV_FILE = f"../data/dataset_aromatic_plants/dataset_{plant_name}.csv"
    
    with open(CSV_FILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["plant_type", "image_path", "date_observed"])
        writer.writerows(csv_rows)

    print("\nDataset et CSV créés avec succès.")
# ---------------------------
# ECRITURE CSV
# ---------------------------
#
#with open(CSV_FILE, "w", newline="") as f:
#    writer = csv.writer(f)
#    writer.writerow(["plant_type", "image_path", "date_observed"])
#    writer.writerows(csv_rows)
#
#print("\nDataset et CSV créés avec succès.")

In [ ]:
# ---------------------------
# TELECHARGEMENT DATASET
# ---------------------------

for plant_name, taxon in plants.items():

    print(f"\nTéléchargement : {plant_name}")

    plant_dir = os.path.join(DATASET_DIR, plant_name)

    os.makedirs(plant_dir, exist_ok=True)

    count = 0
    page = 1

    pbar = tqdm(total=IMAGES_PER_PLANT)

    while count < IMAGES_PER_PLANT:

        url = "https://api.inaturalist.org/v1/observations"

        params = {
            "taxon_name": taxon,
            "quality_grade": "research",
            "identified": "true",
            "photos": "true",
            "per_page": 200,
            "page": page
        }

        response = requests.get(url, params=params).json()

        results = response["results"]

        if not results:
            break

        for obs in results:

            for photo in obs["photos"]:

                img_url = photo["url"].replace("square", "large")

                filename = f"{plant_name}_000{count}.jpg"
                filepath = os.path.join(plant_dir, filename)

                success = download_and_resize(img_url, filepath)

                if success:

                    relative_path = os.path.join(plant_name, filename)

                    csv_rows.append([plant_name, relative_path, date[i]])

                    count += 1
                    pbar.update(1)

                if count >= IMAGES_PER_PLANT:
                    break

            if count >= IMAGES_PER_PLANT:
                break

        page += 1

    pbar.close()


# ---------------------------
# ECRITURE CSV
# ---------------------------

with open(CSV_FILE, "w", newline="") as f:

    writer = csv.writer(f)

    writer.writerow(["plant_type", "image_path", "date_observed"])

    writer.writerows(csv_rows)

print("\nDataset et CSV créés avec succès.")

In [6]:
"""
Scraping d'images d'aromates via DuckDuckGo Images
====================================================
Utilise le package `ddgs` (successeur de `duckduckgo_search`)

Installation :
    pip install ddgs Pillow requests

Arborescence produite :
    dataset_test/
      basilic_commun_0001.jpg
      basilic_commun_0002.jpg
      thym_commun_0001.jpg
      …
      dataset.csv
"""

# ─────────────────────────────────────────────
#  CONFIGURATION
# ─────────────────────────────────────────────

TARGET_COUNT = 10
IMAGE_SIZE   = (400, 400)
OUTPUT_DIR   = "dataset_test"
CSV_FILENAME = "dataset.csv"
DELAY_S      = 0.1        # pause entre téléchargements
TIMEOUT_S    = 10

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

# ─────────────────────────────────────────────
#  30 AROMATES
#  Format : (nom_commun_fr, [requêtes_ddg])
# ─────────────────────────────────────────────
AROMATES: list[tuple[str, list[str]]] = [
    ("basilic commun",          ["basilic", "ocimum basilicum", "basil"]),
    ("thym commun",             ["thym", "thymus vulgaris", "thyme"]),
    ("romarin",                 ["romarin", "rosmarinus officinalis", "rosemary"]),
    ("lavande vraie",           ["lavandula angustifolia", "lavender"]),
    ("menthe poivree",          ["menthe", "mentha piperita", "peppermint"]),
    ("menthe verte",            ["menthe", "mentha spicata", "spearmint"]),
    ("origan",                  ["origan", "origanum vulgare", "oregano"]),
    ("sauge officinale",        ["sauge", "salvia officinalis", "sage"]),
    ("coriandre",               ["coriandre", "coriandrum sativum", "coriander"]),
    ("basilic commun",          ["basilic", "ocimum basilicum", "basil"]),
    ("thym commun",             ["thym", "thymus vulgaris", "thyme"]),
    ("romarin",                 ["romarin", "rosmarinus officinalis", "rosemary"]),
    ("lavande vraie",           ["lavandula angustifolia", "lavender"]),
    ("menthe poivree",          ["menthe", "mentha piperita", "peppermint"]),
    ("menthe verte",            ["menthe", "mentha spicata", "spearmint"]),
    ("origan",                  ["origan", "origanum vulgare", "oregano"]),
    ("sauge officinale",        ["sauge", "salvia officinalis", "sage"]),
    ("coriandre",               ["coriandre", "coriandrum sativum", "coriander"]),
    ("aneth",                   ["aneth", "anethum graveolens", "dill"]),
    ("fenouil",                 ["fenouil", "foeniculum vulgare", "fennel"]),
    ("persil",                  ["persil", "petroselinum crispum", "parsley"]),
    ("cerfeuil",                ["cerfeuil", "anthriscus cerefolium", "chervil"]),
    ("ciboulette",              ["ciboulette", "allium schoenoprasum", "chives"]),
    ("estragon",                ["estragon", "artemisia dracunculus", "tarragon"]),
    ("laurier sauce",           ["laurier sauce", "laurus nobilis", "bay leaf"]),
    ("melisse",                 ["melisse", "melissa officinalis",  "lemon balm"]),
    ("liveche",                 ["liveche", "flevisticum officinale", "lovage"]),
    ("cerfeuil musque",         ["cerfeuil musque", "sweet cicely", "myrrhis odorata"]),
    ("fenugrec",                ["fenugrec", "trigonella foenum-graecum", "fenugreek"]),
    ("carvi",                   ["carvi", "carum carvi", "caraway"]),
    ("anis vert",               ["anis vert", "pimpinella anisum", "anise"]),
    ("cumin",                   ["cumin", "cuminum cyminum", "cumin"]),
    ("hysope officinale",       ["hysope officinale", "hyssopus officinalis", "hyssop"]),
    ("agastache fenouil",       ["agastache fenouil", "agastache foeniculum", "anise hyssop"]),
    ("verveine officinale",     ["verveine officinale", "verbena officinalis", "vervain"]),
    ("bourrache",               ["bourrache", "borago officinalis", "borage"]),
    ("souci officinal",         ["souci officinal", "calendula officinalis", "calendula"]),
    ("camomille romaine",       ["camomille romaine", "chamaemelum nobile", "roman chamomile"]),
]

# ─────────────────────────────────────────────
#  FONCTIONS
# ─────────────────────────────────────────────

def slug(name: str) -> str:
    replacements = {
        "é": "e", "è": "e", "ê": "e", "ë": "e",
        "à": "a", "â": "a", "ä": "a",
        "î": "i", "ï": "i",
        "ô": "o", "ö": "o",
        "ù": "u", "û": "u", "ü": "u",
        "ç": "c", " ": "_", "/": "-",
    }
    result = name.lower()
    for src, dst in replacements.items():
        result = result.replace(src, dst)
    return result


def fetch_image_urls(queries: list[str], count: int) -> list[str]:
    """
    Interroge DuckDuckGo Images via `ddgs` avec fallback progressif.
    Certains filtres (size/type) retournent parfois 0 résultat selon la région.
    """
    urls: list[str] = []
    seen: set[str] = set()

    # On essaie d'abord des recherches strictes, puis plus permissives.
    search_profiles = [
        {"size": "Large", "type_image": "photo"},
        {"size": None, "type_image": "photo"},
        {"size": "Medium", "type_image": None},
        {"size": None, "type_image": None},
    ]

    for query in queries:
        if len(urls) >= count:
            break

        for profile in search_profiles:
            if len(urls) >= count:
                break

            needed = count - len(urls)
            for attempt in range(3):
                try:
                    ddgs = DDGS()
                    results = ddgs.images(
                        query,
                        region="wt-wt",
                        safesearch="moderate",
                        size=profile["size"],
                        type_image=profile["type_image"],
                        max_results=min(needed + 50, 300),
                    )

                    found_in_call = 0
                    for r in results:
                        url = r.get("image") or r.get("thumbnail") or ""
                        if url and url not in seen:
                            urls.append(url)
                            seen.add(url)
                            found_in_call += 1
                        if len(urls) >= count:
                            break

                    if found_in_call > 0:
                        break

                except Exception as e:
                    # No results found, rate limit, etc. On retente avec backoff léger.
                    print(
                        f"    [ERREUR DDG] requête '{query}' (size={profile['size']}, type={profile['type_image']}, essai {attempt + 1}/3) → {e}"
                    )

                time.sleep(1.0 + 0.6 * attempt)

            # petite pause entre profils pour limiter le rate-limit
            time.sleep(0.6)

        # pause entre requêtes (basilic / latin / anglais)
        time.sleep(1.0)

    return urls[:count]


def download_and_resize(url: str, dest_path: str, size: tuple[int, int]) -> bool:
    """Télécharge, crop centré et redimensionne en JPEG 400x400."""
    try:
        resp = requests.get(url, timeout=TIMEOUT_S, headers=HEADERS)
        resp.raise_for_status()

        img = Image.open(BytesIO(resp.content)).convert("RGB")

        w, h = img.size
        target = size[0] / size[1]
        ratio = w / h

        if ratio > target:
            new_w = int(h * target)
            left = (w - new_w) // 2
            img = img.crop((left, 0, left + new_w, h))
        elif ratio < target:
            new_h = int(w / target)
            top = (h - new_h) // 2
            img = img.crop((0, top, w, top + new_h))

        img = img.resize(size, Image.LANCZOS)
        img.save(dest_path, "JPEG", quality=90)
        return True

    except Exception as exc:
        print(f"    [ERREUR DL] {str(exc)[:80]}")
        return False


def process_aromate(
    common_name: str,
    queries: list[str],
    csv_writer,
    index: int,
    total: int,
) -> tuple[int, int]:
    folder_slug = slug(common_name)

    print(f"\n{'═'*60}")
    print(f"[{index}/{total}] {common_name}")
    print(f"  Requêtes : {queries[0]} (+{len(queries)-1} fallbacks)")
    print(f"{'─'*60}")

    print("  Récupération des URLs…")
    urls = fetch_image_urls(queries, TARGET_COUNT)
    print(f"  → {len(urls)} URLs trouvées.")

    if not urls:
        print("  Aucune URL — aromate ignoré.")
        return 0, 0

    success = 0
    for i, url in enumerate(urls, start=1):
        filename = f"{folder_slug}_{i:04d}.jpg"
        dest_path = os.path.join(OUTPUT_DIR, filename)
        rel_path = os.path.join(OUTPUT_DIR, filename)

        print(f"    [{i:>3}/{len(urls)}] {filename} … ", end="", flush=True)

        if download_and_resize(url, dest_path, IMAGE_SIZE):
            csv_writer.writerow({"nom_aromate": common_name, "chemin_image": rel_path})
            success += 1
            print("OK")
        else:
            print("ECHEC")

        time.sleep(DELAY_S)

    return success, len(urls)


# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, CSV_FILENAME)

    total_ok = 0
    total_tried = 0
    start_time = time.time()

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["nom_aromate", "chemin_image"])
        writer.writeheader()

        for idx, (common, queries) in enumerate(AROMATES, start=1):
            ok, tried = process_aromate(common, queries, writer, idx, len(AROMATES))
            total_ok += ok
            total_tried += tried
            f.flush()

    elapsed = int(time.time() - start_time)
    h, m, s = elapsed // 3600, (elapsed % 3600) // 60, elapsed % 60

    print(f"\n{'═'*60}")
    print(f"TERMINÉ — {total_ok}/{total_tried} images téléchargées")
    print(f"Durée totale : {h:02d}h{m:02d}m{s:02d}s")
    print(f"CSV          : {csv_path}")
    print(f"Dossier      : {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()



════════════════════════════════════════════════════════════
[1/38] basilic commun
  Requêtes : basilic (+2 fallbacks)
────────────────────────────────────────────────────────────
  Récupération des URLs…
    [ERREUR DDG] requête 'basilic' (size=Large, type=photo, essai 1/3) → No results found.
  → 10 URLs trouvées.
    [  1/10] basilic_commun_0001.jpg … OK
    [  2/10] basilic_commun_0002.jpg … OK
    [  3/10] basilic_commun_0003.jpg … OK
    [  4/10] basilic_commun_0004.jpg … OK
    [  5/10] basilic_commun_0005.jpg … OK
    [  6/10] basilic_commun_0006.jpg … OK
    [  7/10] basilic_commun_0007.jpg … OK
    [  8/10] basilic_commun_0008.jpg … OK
    [  9/10] basilic_commun_0009.jpg … OK
    [ 10/10] basilic_commun_0010.jpg … OK

════════════════════════════════════════════════════════════
[2/38] thym commun
  Requêtes : thym (+2 fallbacks)
────────────────────────────────────────────────────────────
  Récupération des URLs…
    [ERREUR DDG] requête 'thym' (size=Large, type=photo, essa